In [ ]:
%%writefile PrismTrainKaggle.py

import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchsummary import summary
import torch.distributed as dist
import torch.backends.cudnn as TBC
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.distributed.fsdp import MixedPrecision, StateDictType, FullStateDictConfig
from torch.distributed.fsdp.sharded_grad_scaler import ShardedGradScaler
import cv2
import numpy as np
from sklearn.neighbors import NearestNeighbors
import time
import math
from pathlib import Path
import wandb
import functools
import contextlib

torch._inductor.config.max_autotune = False

torch.manual_seed(7)
torch.cuda.manual_seed_all(7)
TBC.benchmark = True
TBC.deterministic = False
cv2.setNumThreads(0)
StartTime = time.time()
MaxTime = 11.5*3600

NumHeads = 8
HeadDims = 32
EmbedDim = 256

WANDB_PROJECT = user_secrets.get_secret('WANDB_PROJECT')
WANDB_ENTITY = user_secrets.get_secret('WANDB_ENTITY')
WANDB_ARTIFACT_NAME = 'prism-checkpoint'
WANDB_ARTIFACT_DIR = Path('/kaggle/working/wandb_resume')
LATEST_CHECKPOINT_PATH = Path('/kaggle/working/PrismModelLatest.pth')
BEST_CHECKPOINT_PATH = Path('/kaggle/working/PrismModelBest.pth')

class ImageDataset(Dataset):
	def __init__(self,ImgDirName,IsTraining):
		super().__init__()
		self.Buckets = np.load('/kaggle/input/datasets/nkminion/prismtrain/CoordBuckets.npy')
		self.NearestNeighbour = NearestNeighbors(n_neighbors=1,algorithm='auto').fit(self.Buckets)
		self.ImgPaths = []
		self.IsTraining = IsTraining
		for ImgName in os.listdir(os.path.join(ImgDirName,'L')):
			ImgName = os.path.splitext(ImgName)[0]
			LPath = os.path.join(os.path.join(ImgDirName,'L'),ImgName+'.jpg')
			ABPath = os.path.join(os.path.join(ImgDirName,'ab'),ImgName+'.npy')
			if os.path.isfile(LPath) and os.path.isfile(ABPath):
				self.ImgPaths.append((LPath,ABPath))

	def __len__(self):
		return len(self.ImgPaths)
	
	def __getitem__(self, index):
		try:
			InputImg = cv2.imread(self.ImgPaths[index][0],cv2.IMREAD_GRAYSCALE)
			if InputImg is None:
				raise ValueError(f"OpenCV could not read image: {self.ImgPaths[index][0]}")
			
			OutputChannels = np.load(self.ImgPaths[index][1]).astype(np.float32) - 128.0

			#Removed previous image cropping script because dataset has already been cropped

			if self.IsTraining:
				#Horizontal Flip
				if torch.rand(1).item() < 0.5:
					InputImg = cv2.flip(InputImg,1)
					OutputChannels = np.flip(OutputChannels,1)

				#Contrast and Brightness
				if torch.rand(1).item() < 0.3:
					Contrast = 0.8 + (0.4)*torch.rand(1).item()
					Brightness = torch.randint(-20,20,(1,)).item()
					InputImg = np.clip(
						InputImg.astype(np.float32) * Contrast + Brightness,
						0,
						255
					).astype(np.uint8)

			InputChannel = (torch.from_numpy(InputImg).float()/255.0).unsqueeze(2).permute(2,0,1)
			OutputChannels = OutputChannels.reshape(-1,2)
			
			_,Indices = self.NearestNeighbour.kneighbors(OutputChannels)
			OutputTensor = torch.from_numpy(Indices.reshape(256,256)).long()

			return (InputChannel,OutputTensor)
		
		except ValueError:
			RandomFallback = torch.randint(0,len(self.ImgPaths),(1,)).item()
			return self.__getitem__(RandomFallback)

class Attention(nn.Module):
	def __init__(self):
		super().__init__()

		self.QueryMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.KeyMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.ValueMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.OutputMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.Softmax = nn.Softmax(dim=-1)

	def forward(self,x):
		# SeqLen = 256 currently
		BatchSize,SeqLen,_ = x.shape

		QueryOutput = self.QueryMatrix(x)
		KeyOutput = self.KeyMatrix(x)
		ValueOutput = self.ValueMatrix(x)

		# 8 Heads
		QueryOutput = QueryOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		KeyOutput = KeyOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		ValueOutput = ValueOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims

		#Attention(Q, K, V) = (softmax((QK^T)/sqrt(dk)))*V, dk = HeadDims
		KeyOutput = KeyOutput.transpose(2,3) #B,NumHeads,HeadDims,SeqLen
		MatMulOutput = QueryOutput@KeyOutput #B,NumHeads,SeqLen,SeqLen
		MatMulOutput /= math.sqrt(HeadDims)
		MatMulOutput = self.Softmax(MatMulOutput)
		AttentionOutput = MatMulOutput@ValueOutput #B,NumHeads,SeqLen,HeadDims

		AttentionOutput = AttentionOutput.transpose(1,2).contiguous().view(BatchSize,SeqLen,EmbedDim)

		FinalOutput = self.OutputMatrix(AttentionOutput)

		return FinalOutput

class PrismBlock(nn.Module):
	def __init__(self):
		super().__init__()

		self.PseudoQuery = nn.Parameter(torch.randn(EmbedDim)*0.02)
		self.Attention = Attention()
		self.MultiLayerPerceptron = nn.Sequential(
			nn.Linear(in_features=EmbedDim,out_features=EmbedDim*4),
			nn.GELU(),
			nn.Linear(in_features=EmbedDim*4,out_features=EmbedDim)
		)
		self.LayerNorm1 = nn.LayerNorm(EmbedDim)
		self.LayerNorm2 = nn.LayerNorm(EmbedDim)
		self.LayerNorm3 = nn.LayerNorm(EmbedDim)
		self.Softmax = nn.Softmax(dim=0)

	def forward(self,x,HistoryList):
		HistoryList = torch.stack(HistoryList,dim=0) # D,B,SeqLen,EmbedDim
		HistoryListNorm = self.LayerNorm1(HistoryList)

		Scores = (HistoryListNorm*self.PseudoQuery).sum(dim=-1)/math.sqrt(EmbedDim) # D,B,SeqLen
		Weights = self.Softmax(Scores).unsqueeze(-1)

		XRouted = (HistoryList*Weights).sum(dim=0)

		XRoutedNorm = self.LayerNorm2(XRouted)
		x = x + self.Attention(XRoutedNorm)

		xNorm = self.LayerNorm3(x)
		x = x + self.MultiLayerPerceptron(xNorm)

		return x

class PrismModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.HighResConv = nn.Sequential(
			nn.Conv2d(in_channels=1,out_channels=64,kernel_size=3,stride=1,padding=1),
			nn.GELU()
		)
		self.InputLayer = nn.Conv2d(in_channels=64,out_channels=256,kernel_size=16,stride=16) #B,256,16,16
		self.Flatten = nn.Flatten(start_dim=2) #B,256,256
		#Transpose to B,256,256
		self.LearnParam = nn.Parameter(torch.randn((1,256,256)) * 0.02)
		self.ModuleList = nn.ModuleList(
			[PrismBlock() for _ in range(8)]
		)
		self.Decoder1 = nn.Sequential(
			# 16x16 -> 32x32
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GELU(),
			# 32x32 -> 64x64
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GELU(),
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GELU()
		)
		self.Decoder2 = nn.Sequential(
			# 128x128 -> 256x256
			nn.Conv2d(in_channels=320, out_channels=128, kernel_size=3, stride=1, padding=1),
			nn.GELU(),
			# Final map
			nn.Conv2d(in_channels=128, out_channels=313, kernel_size=3, stride=1, padding=1)
		)

	def forward(self,x):
		SkipFeatures = self.HighResConv(x) # B,64,256,256
		x = self.InputLayer(SkipFeatures) # B,256,16,16
		x = self.Flatten(x) # B,256,256
		x = x.transpose(1,2)
		x = x + self.LearnParam # B,256,256

		HistoryList = [x]
		for module in self.ModuleList:
			x = module(x,HistoryList)
			HistoryList.append(x)

		x = x.transpose(1,2).reshape(-1,256,16,16)

		x = self.Decoder1(x)

		x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)

		x = torch.cat([x,SkipFeatures], dim=1) # B,320,256,256

		x = self.Decoder2(x)

		return x

PriorProbs = np.load('/kaggle/input/datasets/nkminion/prismtrain/PriorProbs.npy')
Lambda = 0.1
Q = 313.0
SmoothedProbs = ((1.0-Lambda)*PriorProbs) + (Lambda/Q)
Weights = 1.0/SmoothedProbs
NormalisationFactor = np.sum(PriorProbs*Weights)
Weights /= NormalisationFactor

PrismAutoWrapPolicy = functools.partial(
	transformer_auto_wrap_policy,
	transformer_layer_cls={PrismBlock},
)

MPDType = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MPPolicy = MixedPrecision(
	param_dtype=MPDType,
	reduce_dtype=MPDType,
	buffer_dtype=MPDType,
)

BatchSize = 16
TargetLR = 1e-4
WarmupSteps = 3000

TrainDirName = '/kaggle/input/datasets/pythonicvarun/colorization-dataset/train'
ValDirName = '/kaggle/input/datasets/pythonicvarun/colorization-dataset/val'


def InitWandbRun(rank):
	if rank != 0:
		return None

	InitKwargs = {
		'project': WANDB_PROJECT,
		'resume': 'allow',
		'config': {
			'BatchSize': BatchSize,
			'TargetLR': TargetLR,
			'WarmupSteps': WarmupSteps,
			'NumHeads': NumHeads,
			'HeadDims': HeadDims,
			'EmbedDim': EmbedDim
		}
	}

	if WANDB_ENTITY:
		InitKwargs['entity'] = WANDB_ENTITY

	RunName = os.environ.get('WANDB_RUN_NAME')
	if RunName:
		InitKwargs['name'] = RunName

	try:
		return wandb.init(**InitKwargs)
	except Exception as Error:
		print(f'W&B init failed. Continuing without W&B checkpoint sync. Error: {Error}')
		return None


def DownloadLatestCheckpoint(wandb_run):
	if wandb_run is None:
		return None

	try:
		Artifact = wandb_run.use_artifact(f'{WANDB_ARTIFACT_NAME}:latest', type='model')
		ArtifactDir = Path(Artifact.download(root=str(WANDB_ARTIFACT_DIR)))
		CheckpointPath = ArtifactDir / LATEST_CHECKPOINT_PATH.name

		if CheckpointPath.exists():
			print(f'Downloaded W&B latest checkpoint: {CheckpointPath}')
			return CheckpointPath

		Candidates = list(ArtifactDir.glob('*.pth'))
		if Candidates:
			print(f'Downloaded W&B checkpoint: {Candidates[0]}')
			return Candidates[0]

		print('W&B latest artifact downloaded, but no checkpoint file was found.')
		return None
	except Exception as Error:
		print(f'No W&B latest checkpoint found. Starting from scratch. Error: {Error}')
		return None


def CleanCheckpointStateDict(RawStateDict):
	CleanStateDict = {}
	for key, value in RawStateDict.items():
		CleanKey = key.replace('_orig_mod.', '').replace('module.', '')
		CleanStateDict[CleanKey] = value
	return CleanStateDict


def LoadCheckpointModelState(model, RawStateDict):
	LoadErrors = []
	for CandidateStateDict in (RawStateDict, CleanCheckpointStateDict(RawStateDict)):
		try:
			model.load_state_dict(CandidateStateDict)
			return
		except RuntimeError as Error:
			LoadErrors.append(str(Error))

	raise RuntimeError('Could not load checkpoint model state. Last error: ' + LoadErrors[-1])


def BuildCheckpoint(model, optimiser, scheduler, ImagesSeen, BestLoss, TrainLossHist, ValLossHist, StepCounter, Epoch, PatienceCounter, scaler):
	SavePolicy = FullStateDictConfig(offload_to_cpu=True,rank0_only=True)
	with FSDP.state_dict_type(model,StateDictType.FULL_STATE_DICT, SavePolicy):
		ModelState = model.state_dict()
		OptimiserState = FSDP.optim_state_dict(model,optimiser)

	if int(os.environ['LOCAL_RANK']) == 0:
		return {
			'ModelState': ModelState,
			'OptimiserState': OptimiserState,
			'SchedulerState': scheduler.state_dict(),
			'ScalerState': scaler.state_dict(),
			'ImagesSeen': ImagesSeen,
			'BestLoss': BestLoss,
			'TrainLossHist': TrainLossHist,
			'ValLossHist': ValLossHist,
			'StepCounter': StepCounter,
			'Epoch': Epoch,
			'PatienceCounter': PatienceCounter
		}
	return None


def SaveCheckpoint(CheckpointPath, Checkpoint):
	CheckpointPath.parent.mkdir(parents=True, exist_ok=True)
	torch.save(Checkpoint, CheckpointPath)


def UploadCheckpointArtifact(wandb_run, CheckpointPath, Aliases, Metadata):
	if wandb_run is None:
		return

	try:
		Artifact = wandb.Artifact(WANDB_ARTIFACT_NAME, type='model', metadata=Metadata)
		Artifact.add_file(str(CheckpointPath), name=CheckpointPath.name)
		LoggedArtifact = wandb_run.log_artifact(Artifact, aliases=Aliases)
		LoggedArtifact.wait()
	except Exception as Error:
		print(f'W&B artifact upload failed for {CheckpointPath}. Error: {Error}')


def GetCurrentLR(optimiser):
	return optimiser.param_groups[0]['lr']


def FSDPTrainLoop(Weights,BatchSize,TrainDirName,ValDirName):
	rank = int(os.environ['LOCAL_RANK'])
	GPUCount = int(os.environ['WORLD_SIZE'])
	dist.init_process_group('nccl')
	torch.cuda.set_device(rank)
	device = torch.device(f'cuda:{rank}')
	wandb_run = InitWandbRun(rank)

	ResumeCheckpointPath = None
	if rank == 0:
		ResumeCheckpointPath = DownloadLatestCheckpoint(wandb_run)
	dist.barrier()

	if ResumeCheckpointPath is None:
		for CandidatePath in (WANDB_ARTIFACT_DIR / LATEST_CHECKPOINT_PATH.name, LATEST_CHECKPOINT_PATH):
			if CandidatePath.exists():
				ResumeCheckpointPath = CandidatePath
				break

	model = PrismModel().to(device, non_blocking=True)
	
	if rank == 0:
		print('Model Summary: ')
		summary(model,input_size=(1,256,256))

	Weights = torch.from_numpy(Weights).float().squeeze().to(device, non_blocking=True)
	
	criterion = nn.CrossEntropyLoss(weight=Weights)

	if rank == 0:
		print('Creating Dataset: ')
	TrainDataset = ImageDataset(TrainDirName,True)
	ValDataset = ImageDataset(ValDirName,False)

	TrainSampler = DistributedSampler(TrainDataset,num_replicas=GPUCount,rank=rank,shuffle=True)
	ValSampler = DistributedSampler(ValDataset,num_replicas=GPUCount,rank=rank,shuffle=False)

	TrainLoader = DataLoader(TrainDataset,batch_size=BatchSize,sampler=TrainSampler,num_workers=4,pin_memory=True,prefetch_factor=2,drop_last=True,persistent_workers=True)
	ValLoader = DataLoader(ValDataset,batch_size=BatchSize,sampler=ValSampler,num_workers=4,pin_memory=True,prefetch_factor=2,drop_last=False,persistent_workers=True)

	Checkpoint = None
	if ResumeCheckpointPath is not None and ResumeCheckpointPath.exists():
		if rank == 0:
			print(f'Loaded previous W&B session from {ResumeCheckpointPath}!')
		Checkpoint = torch.load(ResumeCheckpointPath,map_location=device,weights_only=False)
		LoadCheckpointModelState(model, Checkpoint['ModelState'])

	model = FSDP(
		model,
		auto_wrap_policy=PrismAutoWrapPolicy,
		mixed_precision=MPPolicy,
		device_id=rank
	)

	if rank == 0:
		print('Compiling Model...')
	model = torch.compile(model)

	optimiser = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
	scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser,mode='min',factor=0.5,patience=3,threshold=1e-5)
	scaler = ShardedGradScaler()

	if Checkpoint is not None:
		OptimiserState = FSDP.optim_state_dict_to_load(
			model,
			optimiser,
			Checkpoint['OptimiserState']
		)
		optimiser.load_state_dict(OptimiserState)
		scheduler.load_state_dict(Checkpoint['SchedulerState'])
		scaler.load_state_dict(Checkpoint['ScalerState'])
		ImagesSeen = Checkpoint.get('ImagesSeen', 0)
		BestLoss = Checkpoint.get('BestLoss', float('inf'))
		TrainLossHist = Checkpoint.get('TrainLossHist', [])
		ValLossHist = Checkpoint.get('ValLossHist', [])
		StepCounter = Checkpoint.get('StepCounter', 0)
		StartEpoch = Checkpoint.get('Epoch', -1) + 1
		PatienceCounter = Checkpoint.get('PatienceCounter', 0)
	else:
		if rank == 0:
			print('Starting new session')
		ImagesSeen = 0
		BestLoss = float('inf')
		TrainLossHist = []
		ValLossHist = []
		StepCounter = 0
		StartEpoch = 0
		PatienceCounter = 0

	if rank == 0:
		print(f'Starting train session at epoch {StartEpoch+1}...')

	RunningLoss = 0.0
	MaxPatience = 10
	MaxEpochs = 1000
	LastCompletedEpoch = StartEpoch - 1

	try:
		for epoch in range(StartEpoch, MaxEpochs):
			TrainSampler.set_epoch(epoch)
			model.train()
			AccumulationSteps = 4
			optimiser.zero_grad()

			for BatchIdx, (inputs, targets) in enumerate(TrainLoader):
				
				IsSyncStep = (BatchIdx + 1) % AccumulationSteps == 0 or (BatchIdx + 1) == len(TrainLoader)
				
				SyncContext = contextlib.nullcontext() if IsSyncStep else model.no_sync()

				inputs = inputs.to(device, non_blocking=True)
				targets = targets.to(device, non_blocking=True)

				with SyncContext:
					with torch.amp.autocast('cuda'):
						predictions = model(inputs)
						loss = criterion(predictions, targets) / AccumulationSteps
					scaler.scale(loss).backward()

				if IsSyncStep:
					if StepCounter < WarmupSteps:
						CurrentLR = TargetLR * ((StepCounter+1) / WarmupSteps)
						for ParamGroup in optimiser.param_groups:
							ParamGroup['lr'] = CurrentLR

					scaler.unscale_(optimiser)
					model.clip_grad_norm_(1.0)
					
					scaler.step(optimiser)
					scaler.update()
					optimiser.zero_grad()
					
					StepCounter += 1

				BatchLoss = loss.item() * AccumulationSteps
				RunningLoss += BatchLoss
				ImagesSeen += (inputs.size(0) * GPUCount)

				if rank == 0 and wandb_run is not None:
					wandb_run.log({
						'train/batch_loss': BatchLoss,
						'train/running_epoch_loss': RunningLoss / (BatchIdx + 1),
						'train/epoch': epoch + 1,
						'train/batch': BatchIdx + 1,
						'train/epoch_progress': (BatchIdx + 1) / len(TrainLoader),
						'train/learning_rate': GetCurrentLR(optimiser),
						'train/images_seen': ImagesSeen,
						'train/step_counter': StepCounter
					}, step=StepCounter)

			model.eval()
			ValLossTotal = 0.0
			with torch.no_grad():
				for ValImgs,ValTargets in ValLoader:
					ValImgs = ValImgs.to(device,non_blocking=True)
					ValTargets = ValTargets.to(device,non_blocking=True)

					with torch.amp.autocast('cuda'):
						ValPrediction = model(ValImgs)
						ValLoss = criterion(ValPrediction,ValTargets)
					ValLossTotal += ValLoss.item()

					del ValPrediction, ValLoss
					
			LocalAvgLoss = ValLossTotal/len(ValLoader)
			LossTensor = torch.tensor([LocalAvgLoss],device=device)
			dist.all_reduce(LossTensor,op=dist.ReduceOp.SUM)
			GlobalAvgLoss = LossTensor.item()/GPUCount

			LocalAvgTrainLoss = RunningLoss/len(TrainLoader)
			TrainLossTensor = torch.tensor([LocalAvgTrainLoss],device=device)
			dist.all_reduce(TrainLossTensor,op=dist.ReduceOp.SUM)
			AverageTrainLoss = TrainLossTensor.item()/GPUCount

			TrainLossHist.append(AverageTrainLoss)
			ValLossHist.append(GlobalAvgLoss)

			if StepCounter > WarmupSteps:
				scheduler.step(GlobalAvgLoss)

			if rank == 0:
				print(f"Epoch: {epoch+1} | Training Loss: {AverageTrainLoss:.6f} | Validation Loss: {GlobalAvgLoss:.6f}")

			Improved = GlobalAvgLoss < BestLoss
			if Improved:
				BestLoss = GlobalAvgLoss
				PatienceCounter = 0
				if rank == 0:
					print(f"Validation Loss has decreased! Saving best model!")
			else:
				PatienceCounter += 1
				if rank == 0:
					print(f'No improvement... Patience: {PatienceCounter}/{MaxPatience}')

			LastCompletedEpoch = epoch
			Checkpoint = BuildCheckpoint(model, optimiser, scheduler, ImagesSeen, BestLoss, TrainLossHist, ValLossHist, StepCounter, LastCompletedEpoch, PatienceCounter, scaler)

			if rank == 0:
				Metadata = {
					'epoch': LastCompletedEpoch,
					'images_seen': ImagesSeen,
					'step_counter': StepCounter,
					'best_loss': BestLoss,
					'validation_loss': GlobalAvgLoss,
					'training_loss': AverageTrainLoss,
					'patience_counter': PatienceCounter
				}
				SaveCheckpoint(LATEST_CHECKPOINT_PATH, Checkpoint)
				UploadCheckpointArtifact(wandb_run, LATEST_CHECKPOINT_PATH, ['latest'], Metadata)

				if Improved:
					SaveCheckpoint(BEST_CHECKPOINT_PATH, Checkpoint)
					UploadCheckpointArtifact(wandb_run, BEST_CHECKPOINT_PATH, ['best'], Metadata)

				if wandb_run is not None:
					wandb_run.log({
						'epoch': epoch + 1,
						'train_loss': AverageTrainLoss,
						'val_loss': GlobalAvgLoss,
						'best_loss': BestLoss,
						'learning_rate': GetCurrentLR(optimiser),
						'images_seen': ImagesSeen,
						'step_counter': StepCounter,
						'patience_counter': PatienceCounter,
						'epoch/train_loss': AverageTrainLoss,
						'epoch/val_loss': GlobalAvgLoss,
						'epoch/best_loss': BestLoss,
						'epoch/learning_rate': GetCurrentLR(optimiser),
						'epoch/images_seen': ImagesSeen,
						'epoch/patience_counter': PatienceCounter
					}, step=StepCounter)

			if (PatienceCounter >= MaxPatience) and (StepCounter > WarmupSteps):
				if rank == 0:
					print('Patience limit reached. Stopping...')
				break
			
			RunningLoss = 0.0
			model.train()
			
			StopFlag = torch.zeros(1, device=device)

			if rank == 0 and (time.time() - StartTime) > MaxTime:
				StopFlag += 1

			dist.all_reduce(StopFlag, op=dist.ReduceOp.MAX)

			if StopFlag.item() > 0:
				if rank == 0:
					print('Approaching time limit!')
				raise KeyboardInterrupt

	except KeyboardInterrupt:
		if rank == 0:
			print(f"Keyboard Interrupt! Saving latest model!")
			Checkpoint = BuildCheckpoint(model, optimiser, scheduler, ImagesSeen, BestLoss, TrainLossHist, ValLossHist, StepCounter, LastCompletedEpoch, PatienceCounter, scaler)
			Metadata = {
				'epoch': LastCompletedEpoch,
				'images_seen': ImagesSeen,
				'step_counter': StepCounter,
				'best_loss': BestLoss,
				'patience_counter': PatienceCounter,
				'interrupted': True
			}
			SaveCheckpoint(LATEST_CHECKPOINT_PATH, Checkpoint)
			UploadCheckpointArtifact(wandb_run, LATEST_CHECKPOINT_PATH, ['latest'], Metadata)
	finally:
		if rank == 0 and wandb_run is not None:
			wandb_run.finish()
		dist.destroy_process_group()

if __name__ == '__main__':
	FSDPTrainLoop(Weights,BatchSize,TrainDirName,ValDirName)

In [ ]:
!PYTHONUNBUFFERED=1 torchrun --nproc_per_node=2 PrismTrainKaggle.py 2>&1 | tee PrismTrainLog.txt